# Notebook 07 — Array Scale Specifies Control Constraints

**Engineering statement:** Large neutral-atom arrays become scalable computational platforms only when filling, transport, and control preserve coherence.

Notebook 07 develops the first computation branch of `scalable-platforms`: optical tweezer arrays for neutral-atom computation.

The notebook asks four engineering questions:

- How does useful capacity scale?
- How much coherence is consumed by transport?
- Why does parallel control matter?
- Which engineering resource remains for logical computation?

Generated figures:

```text
07_hero_statement.png
07_array_scaling_chain.png
07_tweezer_capacity.png
07_transport_overhead.png
07_parallel_control.png
07_engineering_budget.png
```


## 1. Setup

This notebook is standalone and works in Colab, Jupyter, or a local repo clone. It writes PNGs to:

```text
figures/
results/07_optical_tweezer_scaling/figures/
```


In [ ]:
from pathlib import Path
import json
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_NAME = "07_optical_tweezer_scaling"
CWD = Path.cwd().resolve()

if (CWD / "notebooks").exists() or (CWD / "figures").exists():
    REPO_ROOT = CWD
elif CWD.name == "notebooks":
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / NOTEBOOK_NAME
RESULT_FIGURES = RESULTS / "figures"

for path in [NOTEBOOKS, FIGURES, RESULTS, RESULT_FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

def save_png(fig, name, dpi=220):
    repo_path = FIGURES / f"{name}.png"
    result_path = RESULT_FIGURES / f"{name}.png"
    fig.savefig(repo_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(result_path, dpi=dpi, bbox_inches="tight")
    print(f"✓ Saved {repo_path}")
    print(f"✓ Saved {result_path}")
    return repo_path

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 2. Variables and abstractions

Notebook 07 uses a compact scaling vocabulary.

\[
N_{\mathrm{atoms}} = f N_{\mathrm{sites}}
\]

\[
R_{\mathrm{control}}
=
\frac{N_{\mathrm{move}}T_{\mathrm{move}}}{T_{\mathrm{coh}}}
\]

where \(N_{\mathrm{sites}}\) is the number of tweezer sites, \(f\) is filling fraction, \(N_{\mathrm{move}}\) is the number of moved atoms, \(T_{\mathrm{move}}\) is move time, and \(T_{\mathrm{coh}}\) is the coherence budget.


In [ ]:
scaling_variables = pd.DataFrame([
    {"symbol": "N_sites", "meaning": "available optical tweezer sites", "role": "array scale"},
    {"symbol": "N_atoms", "meaning": "trapped neutral atoms", "role": "available qubits"},
    {"symbol": "f", "meaning": "filling fraction", "role": "site occupancy"},
    {"symbol": "N_move", "meaning": "atoms moved during rearrangement", "role": "transport workload"},
    {"symbol": "T_move", "meaning": "time per move operation", "role": "transport latency"},
    {"symbol": "T_coh", "meaning": "useful coherence time", "role": "coherence budget"},
    {"symbol": "R_control", "meaning": "control overhead", "role": "scaling constraint"},
])
variables_path = RESULTS / "scaling_variables.csv"
scaling_variables.to_csv(variables_path, index=False)
scaling_variables


## 3. Hero statement

Array scale specifies control constraints.


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 7.2))
ax.axis("off")

ax.text(
    0.5, 0.78,
    "Array Scale Specifies\nControl Constraints",
    ha="center", va="center",
    fontsize=34,
    weight="bold",
)

ax.text(
    0.5, 0.56,
    "Optical tweezer arrays scale computation by scaling\nneutral atoms, filling, transport, and control.",
    ha="center", va="center",
    fontsize=17,
)

ax.text(
    0.5, 0.36,
    "array scale  →  filling fraction  →  rearrangement  →  coherent transport  →  error correction",
    ha="center", va="center",
    fontsize=14,
    bbox=dict(boxstyle="round,pad=0.65", linewidth=1.2, facecolor="white"),
)

ax.text(
    0.5, 0.18,
    "large arrays  •  high filling  •  low overhead  •  long coherence",
    ha="center", va="center",
    fontsize=14,
)

fig.tight_layout()
hero_fig = save_png(fig, "07_hero_statement")
plt.show()
hero_fig


## 4. Scaling pipeline

A large array becomes useful only if the control system preserves coherence while scaling the number of sites and atoms.


In [ ]:
chain_steps = [
    "Optical\ntweezer sites",
    "Loaded\natoms",
    "Filling\nfraction",
    "Rearrangement",
    "Coherent\ntransport",
    "Logical\nlayout",
    "Error\ncorrection",
]

fig, ax = plt.subplots(figsize=(13.5, 3.4))
ax.axis("off")
y = 0.5

for i, label in enumerate(chain_steps):
    ax.text(
        i, y, label,
        ha="center", va="center",
        fontsize=11.5,
        bbox=dict(boxstyle="round,pad=0.52", linewidth=1.2, facecolor="white"),
    )
    if i < len(chain_steps) - 1:
        ax.annotate(
            "",
            xy=(i + 0.68, y),
            xytext=(i + 0.32, y),
            arrowprops=dict(arrowstyle="->", linewidth=1.6),
        )

ax.set_xlim(-0.6, len(chain_steps) - 0.4)
ax.set_ylim(0, 1)
ax.set_title("Array scale becomes useful through control", fontsize=16)
fig.tight_layout()

chain_fig = save_png(fig, "07_array_scaling_chain")
plt.show()
chain_fig


## 5. Capacity

Capacity depends on both array size and filling fraction.

\[
N_{\mathrm{atoms}} = fN_{\mathrm{sites}}
\]


In [ ]:
site_counts = [500, 1000, 2000, 4000, 6000, 8000, 12000]
fillings = [0.35, 0.50, 0.65, 0.80, 0.90]

capacity_rows = []
for n in site_counts:
    for f in fillings:
        capacity_rows.append({
            "N_sites": n,
            "filling_fraction": f,
            "N_atoms": f * n,
        })

capacity = pd.DataFrame(capacity_rows)
capacity_path = RESULTS / "tweezer_capacity.csv"
capacity.to_csv(capacity_path, index=False)

fig, ax = plt.subplots(figsize=(9, 5.2))

for f in fillings:
    sub = capacity[capacity["filling_fraction"] == f]
    ax.plot(sub["N_sites"], sub["N_atoms"], marker="o", label=f"f = {f:.2f}")

ax.set_xlabel("Tweezer sites")
ax.set_ylabel("Trapped atoms")
ax.set_title("Capacity depends on both array size and filling fraction")
ax.legend(title="Filling")
fig.tight_layout()

capacity_fig = save_png(fig, "07_tweezer_capacity")
plt.show()
capacity.head()


## 6. Transport overhead

Rearrangement and long-distance transport are useful only when their overhead fits inside the coherence budget.

\[
R_{\mathrm{control}}
=
\frac{N_{\mathrm{move}}T_{\mathrm{move}}}{T_{\mathrm{coh}}}
\]

Lower \(R_{\mathrm{control}}\) means the platform spends a smaller fraction of its coherence budget on rearrangement.


In [ ]:
T_coh = 10.0  # seconds, illustrative coherence budget
move_counts = [50, 100, 200, 500, 1000, 2000, 4000]
move_times_ms = [0.05, 0.10, 0.25, 0.50, 1.00]

overhead_rows = []
for nmove in move_counts:
    for t_ms in move_times_ms:
        overhead_rows.append({
            "N_move": nmove,
            "T_move_ms": t_ms,
            "T_coh_s": T_coh,
            "R_control": nmove * (t_ms / 1000.0) / T_coh,
        })

overhead = pd.DataFrame(overhead_rows)
overhead_path = RESULTS / "transport_overhead.csv"
overhead.to_csv(overhead_path, index=False)

fig, ax = plt.subplots(figsize=(9, 5.2))

for t_ms in move_times_ms:
    sub = overhead[overhead["T_move_ms"] == t_ms]
    ax.plot(sub["N_move"], sub["R_control"], marker="o", label=f"{t_ms:g} ms/move")

ax.axhline(0.1, linestyle="--", linewidth=1.2)
ax.text(max(move_counts) * 0.72, 0.105, "10% coherence budget", fontsize=10)
ax.set_xlabel("Moved atoms")
ax.set_ylabel("Control overhead")
ax.set_title("Transport overhead competes with coherence budget")
ax.legend(title="Move time")
fig.tight_layout()

overhead_fig = save_png(fig, "07_transport_overhead")
plt.show()
overhead.head()


## 7. Parallel control

Large arrays become computational platforms when many coherent operations execute simultaneously.


In [ ]:
parallel_pipeline = [
    ("Larger arrays", "more sites and atoms"),
    ("Independent\ncontrol regions", "local operations"),
    ("Simultaneous\ngate operations", "parallel execution"),
    ("Higher logical\nthroughput", "more useful work"),
    ("Fault-tolerant\ncomputation", "error-corrected pathway"),
]

fig, ax = plt.subplots(figsize=(10.5, 7.2))
ax.axis("off")

ax.text(0.5, 0.95, "Parallel control specifies useful scale", ha="center", va="center", fontsize=20, weight="bold")
ax.text(
    0.5, 0.88,
    "Large neutral-atom arrays become computational platforms when many coherent operations execute simultaneously.",
    ha="center", va="center", fontsize=12.5,
)

x = 0.5
y0 = 0.76
dy = 0.125

for i, (title, subtitle) in enumerate(parallel_pipeline):
    y = y0 - i * dy
    ax.text(
        x, y,
        f"{title}\n{subtitle}",
        ha="center", va="center",
        fontsize=12.5,
        bbox=dict(boxstyle="round,pad=0.55", linewidth=1.2, facecolor="white"),
    )
    if i < len(parallel_pipeline) - 1:
        ax.annotate(
            "",
            xy=(x, y - 0.075),
            xytext=(x, y - 0.045),
            arrowprops=dict(arrowstyle="->", linewidth=1.4),
        )

ax.text(
    0.5, 0.08,
    "Useful scale is determined by parallel coherent control rather than atom count alone.",
    ha="center", va="center", fontsize=13,
)

fig.tight_layout()
parallel_fig = save_png(fig, "07_parallel_control")
plt.show()
parallel_fig


## 8. Engineering budget

The notebook ends with the resource that motivates Notebook 13.

Every control operation consumes part of the available coherent window:

\[
T_{\mathrm{coh}}
\rightarrow
T_{\mathrm{transport}}
+
T_{\mathrm{address}}
+
T_{\mathrm{gate}}
+
T_{\mathrm{measure}}
+
T_{\mathrm{margin}}
\]


In [ ]:
budget_rows = [
    {"stage": "Total coherence", "role": "available coherent window"},
    {"stage": "Transport", "role": "rearrangement and long-distance motion"},
    {"stage": "Addressing", "role": "site-selective control and crosstalk"},
    {"stage": "Gate operations", "role": "entangling and logical operations"},
    {"stage": "Measurement", "role": "readout and verification"},
    {"stage": "Remaining budget", "role": "margin for logical fidelity"},
]
budget = pd.DataFrame(budget_rows)
budget_path = RESULTS / "coherence_budget.csv"
budget.to_csv(budget_path, index=False)

budget_steps = [
    ("Total\ncoherence", "available window"),
    ("Transport", "move atoms"),
    ("Addressing", "select sites"),
    ("Gate\noperations", "entangle"),
    ("Measurement", "read out"),
    ("Remaining\nbudget", "logical fidelity"),
]

fig, ax = plt.subplots(figsize=(12.5, 6.2))
ax.axis("off")

ax.text(0.5, 0.93, "Coherence budget for optical tweezer control", ha="center", va="center", fontsize=19, weight="bold")
ax.text(0.5, 0.85, "Every control operation consumes part of the available coherent window.", ha="center", va="center", fontsize=13)

for i, (title, subtitle) in enumerate(budget_steps):
    x = 0.09 + i * 0.165
    ax.text(
        x, 0.58,
        title,
        ha="center", va="center",
        fontsize=12.5,
        weight="bold",
        bbox=dict(boxstyle="round,pad=0.52", linewidth=1.2, facecolor="white"),
    )
    ax.text(x, 0.37, subtitle, ha="center", va="center", fontsize=10.5)
    if i < len(budget_steps) - 1:
        ax.annotate(
            "",
            xy=(x + 0.105, 0.58),
            xytext=(x + 0.055, 0.58),
            arrowprops=dict(arrowstyle="->", linewidth=1.3),
        )

ax.text(
    0.5, 0.14,
    "Notebook 13 asks how hyperfine coherence enlarges the available engineering budget.",
    ha="center", va="center",
    fontsize=12.5,
)

fig.tight_layout()
budget_fig = save_png(fig, "07_engineering_budget")
plt.show()
budget


## 9. Takeaways

Array scale specifies control constraints because useful computation depends on preserving coherence while increasing parallel control.

Notebook 13 begins with the remaining engineering resource: hyperfine coherence.


In [ ]:
summary = pd.DataFrame([
    {"takeaway": "Array scale expands possible layouts."},
    {"takeaway": "Filling fraction determines usable capacity."},
    {"takeaway": "Transport consumes coherence budget."},
    {"takeaway": "Parallel control converts large arrays into useful computation."},
    {"takeaway": "Hyperfine coherence becomes the next engineering resource."},
])
summary_path = RESULTS / "notebook_07_takeaways.csv"
summary.to_csv(summary_path, index=False)
summary


## 10. Save notebook manifest


In [ ]:
notebook_manifest = {
    "notebook": "07_array_scale_specifies_control_constraints.ipynb",
    "title": "Array Scale Specifies Control Constraints",
    "primary_specification": "array scale specifies control constraints",
    "statement": "Large neutral-atom arrays become scalable computational platforms only when filling, transport, and control preserve coherence.",
    "outputs": {
        "scaling_variables": str(variables_path.relative_to(REPO_ROOT)),
        "capacity_csv": str(capacity_path.relative_to(REPO_ROOT)),
        "overhead_csv": str(overhead_path.relative_to(REPO_ROOT)),
        "budget_csv": str(budget_path.relative_to(REPO_ROOT)),
        "summary_csv": str(summary_path.relative_to(REPO_ROOT)),
        "hero_figure": str(hero_fig.relative_to(REPO_ROOT)),
        "chain_figure": str(chain_fig.relative_to(REPO_ROOT)),
        "capacity_figure": str(capacity_fig.relative_to(REPO_ROOT)),
        "overhead_figure": str(overhead_fig.relative_to(REPO_ROOT)),
        "parallel_figure": str(parallel_fig.relative_to(REPO_ROOT)),
        "budget_figure": str(budget_fig.relative_to(REPO_ROOT)),
    },
}

manifest_path = RESULTS / "07_array_scale_specifies_control_constraints_manifest.json"
manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
notebook_manifest


## 11. Download artifacts

Run the final cell to package the notebook outputs for download.


In [ ]:
from IPython.display import FileLink, display

zip_path = RESULTS / "07_array_scale_specifies_control_constraints_artifacts.zip"

paths_to_include = [
    FIGURES,
    RESULTS,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
